# S1-S5 OLS 점검

이 노트북은 스마트팜 센서 데이터에서 **S1(전체 구간 레벨 OLS)**, **S2(펌프 작동 구간 + HAC + 1차 차분)**, **S3(VIF 정리용 축소 모델 비교)**, **S5(lag feature 스캔)** 을 순서대로 점검하기 위한 작업 노트입니다.

핵심 원칙은 다음과 같습니다.

1. `S1`은 빠른 스크리닝용 베이스라인입니다. 높은 `R2`가 나와도 그대로 믿지 않습니다.
2. `S2`는 펌프가 실제로 도는 구간만 남기고, 자기상관에 덜 취약한 HAC 표준오차와 1차 차분을 함께 점검합니다.
3. `DW`가 매우 낮거나 `VIF`가 매우 크면, 그 모델은 좋은 설명모형이라기보다 추세나 상태전이를 따라간 결과일 수 있습니다.
4. 비율형 타깃(`zone1_resistance`, `wire_to_water_efficiency`)은 정의식의 영향이 남기 때문에 더 보수적으로 해석합니다.
5. `S3`는 `DIFF_on`에서도 공선성이 큰 타깃만 다시 다루며, 변수 제거가 정말 정당한지는 `AIC/BIC` 손실까지 같이 봅니다.
6. `S5`는 lag를 넣어 잔차 자기상관을 줄일 수 있는지 보는 단계입니다. 단, `DW`가 좋아져도 `VIF`가 급등하면 계수 해석은 여전히 조심해야 합니다.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 입력 CSV 후보 경로입니다. 앞에서부터 존재하는 경로를 사용합니다.
INPUT_PATHS = [
    Path("human_A/src/selected_smartfarm.csv"),
    Path("src/selected_smartfarm.csv"),
    Path("selected_smartfarm.csv"),
]

# 결과 CSV 파일명입니다. 실제 저장 위치는 실행 폴더에 맞춰 아래 helper에서 결정합니다.
OUTPUT_S1_NAME = "ols_first_pass.csv"
OUTPUT_S2_NAME = "ols_first_pass_v2.csv"
OUTPUT_S3_NAME = "ols_s3_vif_cleanup.csv"
OUTPUT_S5_SCAN_NAME = "ols_s5_lag_scan.csv"
OUTPUT_S5_SUMMARY_NAME = "ols_s5_lag_summary.csv"

# 타깃별 설명변수 사전입니다.
# 아래 주석에는 왜 넣었는지, 왜 뺐는지를 함께 기록해 둡니다.
TARGET_DICTIONARY = {
    # ============================================================
    # 1) motor_current_a
    # ============================================================
    "motor_current_a": [
        # "motor_power_kw",         # ❌ 제외: P=V·I 물리 항등식이라 전류와 직접 묶입니다.
        #                           #    회귀라기보다 공식을 다시 확인하는 셈이어서 제외합니다.
        # "wire_to_water_efficiency", # ❌ 제외: 파생변수이며 motor_power_kw를 분모로 포함합니다.
        #                           #    target과 간접 순환 구조가 생겨 해석이 흐려집니다.
        "motor_temperature_c",      # ✅ 유지: 권선 저항과 발열을 반영하는 비교적 독립적인 원인 후보입니다.
        "pump_rpm",                 # ✅ 유지: 외생적 운전점 제어 변수로 해석 가능합니다.
        "discharge_pressure_kpa",   # ✅ 유지: 펌프 부하를 대표하는 물리량입니다.
    ],

    # ============================================================
    # 2) zone1_resistance (= zone1_pressure / zone1_flow)
    # ============================================================
    "zone1_resistance": [
        "zone1_pressure_kpa",       # ⚠️ 유지: 정의식의 분자입니다.
        #                           #    이 모델은 순수 예측이 아니라 분자 기여도를 보는 용도로 해석합니다.
        "zone1_flow_l_min",         # ⚠️ 유지: 정의식의 분모입니다.
        #                           #    따라서 계수 해석은 '저항 예측'보다 '분자·분모 분해'에 가깝습니다.
    ],

    # ============================================================
    # 3) wire_to_water_efficiency (= hydraulic_power / motor_power)
    # ============================================================
    "wire_to_water_efficiency": [
        # "motor_power_kw",         # ❌ 제외: 효율 정의식의 분모라 직접적인 공선성 원인입니다.
        # "hydraulic_power_kw",     # ❌ 제외: 효율 정의식의 분자라 직접적인 공선성 원인입니다.
        # "differential_pressure_kpa", # ❌ 제외: hydraulic_power와 매우 강하게 얽힐 가능성이 큽니다.
        # "flow_rate_l_min",        # ❌ 제외: hydraulic_power 구성 성분이어서 정의식 오염이 남습니다.
        "pump_rpm",                 # ✅ 유지: 운전점 변화 자체를 반영하는 외생 변수입니다.
        "motor_temperature_c",      # ✅ 유지: 전기적/열적 손실을 반영합니다.
        "suction_pressure_kpa",     # ✅ 유지: 흡입 상태와 캐비테이션 징후를 간접 반영합니다.
        "bearing_temperature_c",    # ✅ 유지: 기계적 손실과 마찰 증가를 반영합니다.
    ],

    # ============================================================
    # 4) mix_ph
    # ============================================================
    "mix_ph": [
        # "mix_target_ph",          # ❌ 제외: 설정값이 사실상 상수라 설명변수 역할을 못합니다.
        "dosing_acid_ml_min",       # ✅ 유지: pH를 직접 바꾸는 주입량입니다.
        "mix_temp_c",               # ✅ 유지: pH 센서와 용액 반응의 온도 의존성을 반영합니다.
        "acid_tank_level_pct",      # ✅ 유지: 산액 부족 시 제어 실패 가능성을 반영합니다.
        "mix_flow_l_min",           # ✅ 유지: 희석 및 혼합량 변화를 반영합니다.
    ],

    # ============================================================
    # 5) mix_ec_ds_m
    # ============================================================
    "mix_ec_ds_m": [
        "mix_target_ec_ds_m",       # ✅ 유지: 이 변수는 실제로 값이 변하며 설정 농도를 반영합니다.
        "tank_a_level_pct",         # ✅ 유지: A 비료 잔량 변화가 공급 농도에 영향을 줄 수 있습니다.
        "tank_b_level_pct",         # ✅ 유지: B 비료 잔량 변화도 같은 이유로 포함합니다.
        "mix_temp_c",               # ✅ 유지: 전도도는 온도 영향이 비교적 큽니다.
        "mix_flow_l_min",           # ✅ 유지: 희석 정도를 반영합니다.
        "raw_water_temp_c",         # ✅ 유지: 원수 온도에 따른 배경 변화를 반영합니다.
    ],
}

# 비율형 타깃은 정의식의 영향을 직접 받으므로 해석 메모를 따로 둡니다.
TARGET_NOTES = {
    "zone1_resistance": "분자·분모 기여도 분해 관점으로만 해석해야 합니다.",
    "wire_to_water_efficiency": "운전점과 정의식 영향이 함께 남는 비율형 지표입니다.",
}


In [ ]:
def resolve_existing_path(paths):
    """경로 후보 중 실제로 존재하는 첫 파일 경로를 반환합니다."""
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(paths)


def get_analysis_home():
    """노트북이 어느 폴더에서 실행되든 human_A 작업 폴더를 찾아 반환합니다."""
    cwd = Path.cwd()
    if (cwd / "src").exists() and (cwd / "H2_preregistered_spec.md").exists():
        return cwd
    if (cwd / "human_A").exists() and (cwd / "human_A" / "src").exists():
        return cwd / "human_A"
    return cwd


def resolve_output_path(filename):
    """실행 위치와 무관하게 결과 CSV를 human_A 폴더 기준으로 저장할 경로를 반환합니다."""
    return get_analysis_home() / filename


def resolve_existing_output_path(filename):
    """이미 저장된 결과 CSV를 실행 위치와 무관하게 찾아 반환합니다."""
    analysis_home = get_analysis_home()
    candidates = [
        analysis_home / filename,
        Path(filename),
        Path("human_A") / filename,
    ]
    return resolve_existing_path(candidates)


def load_source_frame(path):
    """원본 CSV를 불러와 timestamp 기준으로 정렬합니다."""
    df = pd.read_csv(path, parse_dates=["timestamp"])
    return df.sort_values("timestamp")


def add_derived_columns(df):
    """분석에 필요한 파생변수를 생성합니다."""
    out = df.copy()
    out["hydraulic_power_kw"] = out["discharge_pressure_kpa"] * out["flow_rate_l_min"] / 60000
    out["differential_pressure_kpa"] = out["discharge_pressure_kpa"] - out["suction_pressure_kpa"]
    out["wire_to_water_efficiency"] = out["hydraulic_power_kw"] / out["motor_power_kw"].replace(0, np.nan)
    out["zone1_resistance"] = out["zone1_pressure_kpa"] / out["zone1_flow_l_min"].replace(0, np.nan)
    return out.replace([np.inf, -np.inf], np.nan)


def compute_vif_table(X):
    """설명변수별 VIF를 계산합니다."""
    if X.shape[1] == 0:
        return {}
    if X.shape[1] == 1:
        return {X.columns[0]: 1.0}

    values = X.astype(float).values
    return {
        column: float(variance_inflation_factor(values, idx))
        for idx, column in enumerate(X.columns)
    }


def build_quality_flags(r2, dw, max_vif):
    """적합 결과를 빠르게 읽기 위한 경고 문구를 만듭니다."""
    flags = []
    if dw < 1.0:
        flags.append("잔차 자기상관 심함")
    elif dw < 1.5:
        flags.append("잔차 자기상관 주의")

    if max_vif > 30:
        flags.append("다중공선성 매우 큼")
    elif max_vif > 10:
        flags.append("다중공선성 큼")

    if r2 > 0.95 and dw < 0.5:
        flags.append("추세 또는 상태전이 주도 적합 의심")

    return " | ".join(flags) if flags else "특이 경고 없음"


def build_target_note(target):
    """타깃별 해석 메모를 반환합니다."""
    return TARGET_NOTES.get(target, "물리적 연결과 운전 정책 영향을 함께 확인해야 합니다.")


def run_ols_s1(df, y_col, x_cols, tag="S1_LEVEL_all"):
    """S1용 전체 구간 레벨 OLS를 실행합니다."""
    d = df[[y_col] + x_cols].dropna()
    if d.empty:
        raise ValueError(f"dropna 이후 남은 행이 없습니다: {y_col}")

    X = sm.add_constant(d[x_cols], has_constant="add")
    res = sm.OLS(d[y_col], X).fit()
    dw = sm.stats.stattools.durbin_watson(res.resid)
    jb_stat, jb_p, _, _ = sm.stats.stattools.jarque_bera(res.resid)
    vif = compute_vif_table(d[x_cols])
    max_vif = float(max(vif.values())) if vif else np.nan

    return {
        "target": y_col,
        "variant": tag,
        "sample_scope": "전체 시점",
        "cov_type": "기본 OLS",
        "N": int(res.nobs),
        "R2": float(res.rsquared),
        "AdjR2": float(res.rsquared_adj),
        "F": float(res.fvalue),
        "F_p": float(res.f_pvalue),
        "AIC": float(res.aic),
        "BIC": float(res.bic),
        "DW": float(dw),
        "JB_p": float(jb_p),
        "JB_stat": float(jb_stat),
        "max_vif": max_vif,
        "quality_flags": build_quality_flags(float(res.rsquared), float(dw), max_vif),
        "target_note": build_target_note(y_col),
        "predictors": x_cols,
        "coef": {key: float(value) for key, value in res.params.drop("const", errors="ignore").items()},
        "pvals": {key: float(value) for key, value in res.pvalues.drop("const", errors="ignore").items()},
        "vif": vif,
    }


def diagnose_pump_on_rules(df, rpm_thresh=100, flow_thresh=1.0):
    """펌프 작동 구간 정의를 점검하고 기본 마스크를 반환합니다."""
    if "pump_rpm" in df.columns:
        rpm_mask = df["pump_rpm"] > rpm_thresh
        selected_mask = rpm_mask
        selected_rule = f"pump_rpm > {rpm_thresh}"
    else:
        rpm_mask = pd.Series(False, index=df.index)
        selected_mask = df["flow_rate_l_min"] > flow_thresh
        selected_rule = f"flow_rate_l_min > {flow_thresh}"

    flow_mask = df["flow_rate_l_min"] > flow_thresh
    mismatch_mask = rpm_mask != flow_mask if "pump_rpm" in df.columns else pd.Series(False, index=df.index)

    diagnostics = {
        "선택한 기준": selected_rule,
        "전체 행 수": int(len(df)),
        "pump_rpm 기준 on 행 수": int(rpm_mask.sum()) if "pump_rpm" in df.columns else np.nan,
        "flow_rate 기준 on 행 수": int(flow_mask.sum()),
        "불일치 행 수": int(mismatch_mask.sum()) if "pump_rpm" in df.columns else 0,
    }
    mismatch_preview = df.loc[mismatch_mask, ["timestamp", "pump_rpm", "flow_rate_l_min"]].head(10) if mismatch_mask.any() else pd.DataFrame()
    return selected_mask, diagnostics, mismatch_preview


def run_ols_s2(df, y_col, x_cols, tag, hac_lags=20):
    """S2용 HAC 표준오차 OLS를 실행합니다."""
    d = df[[y_col] + x_cols].dropna()
    if len(d) < 100:
        print(f"[{tag}] 건너뜀: N={len(d)} 부족")
        return None

    X = sm.add_constant(d[x_cols], has_constant="add")
    res = sm.OLS(d[y_col], X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    dw = sm.stats.stattools.durbin_watson(res.resid)
    jb_stat, jb_p, _, _ = sm.stats.stattools.jarque_bera(res.resid)
    vif = compute_vif_table(d[x_cols])
    max_vif = float(max(vif.values())) if vif else np.nan

    return {
        "target": y_col,
        "variant": tag,
        "sample_scope": "펌프 작동 구간" if tag == "LEVEL_on" else "펌프 작동 구간 1차 차분",
        "cov_type": f"HAC(maxlags={hac_lags})",
        "N": int(res.nobs),
        "R2": float(res.rsquared),
        "AdjR2": float(res.rsquared_adj),
        "AIC": float(res.aic),
        "BIC": float(res.bic),
        "DW": float(dw),
        "JB_p": float(jb_p),
        "JB_stat": float(jb_stat),
        "max_vif": max_vif,
        "quality_flags": build_quality_flags(float(res.rsquared), float(dw), max_vif),
        "target_note": build_target_note(y_col),
        "coef": {key: float(value) for key, value in res.params.drop("const", errors="ignore").items()},
        "pvals": {key: float(value) for key, value in res.pvalues.drop("const", errors="ignore").items()},
        "vif": vif,
    }


def recommend_variant(level_row, diff_row):
    """S2 두 변형 중 어느 쪽을 우선 볼지 추천하되, 남은 문제도 함께 표시합니다."""
    if level_row is None or diff_row is None:
        return "비교 불가", "필수 결과가 비어 있습니다."

    level_score = abs(level_row["DW"] - 2.0) + max(level_row["max_vif"] - 10.0, 0.0) / 10.0
    diff_score = abs(diff_row["DW"] - 2.0) + max(diff_row["max_vif"] - 10.0, 0.0) / 10.0

    if diff_score <= level_score:
        candidate_name = "DIFF_on"
        candidate_row = diff_row
        base_reason = "차분 후 DW가 2에 더 가까워지고 공선성도 줄었습니다."
    else:
        candidate_name = "LEVEL_on"
        candidate_row = level_row
        base_reason = "레벨 모델이 상대적으로 더 안정적입니다."

    if candidate_row["DW"] < 0.8 or candidate_row["max_vif"] > 30:
        return "추가 재설계 필요", base_reason + " 다만 자기상관 또는 공선성 문제가 여전히 커서 바로 해석하기 어렵습니다."

    if candidate_row["DW"] < 1.5 or candidate_row["max_vif"] > 10:
        return f"{candidate_name} (주의)", base_reason + " 다만 잔차 자기상관이나 공선성 문제가 일부 남아 있습니다."

    return candidate_name, base_reason + " 현재 진단상 가장 해석 가능한 후보입니다."


def judge_s3_candidate(base_row, candidate_row):
    """S3 축소 모델이 공선성 해소 대비 설명력 손실을 감수할 만한지 판정합니다."""
    delta_aic = float(candidate_row["AIC"] - base_row["AIC"])
    delta_bic = float(candidate_row["BIC"] - base_row["BIC"])
    candidate_max_vif = float(candidate_row["max_vif"])
    candidate_dw = float(candidate_row["DW"])

    if delta_aic <= -2 and candidate_max_vif < 4 and candidate_dw >= 1.5:
        verdict = "채택 후보"
        reason = "AIC가 개선되고 VIF와 DW도 함께 좋아졌습니다."
    elif delta_aic <= -2 and candidate_max_vif < 4:
        verdict = "조건부 후보"
        reason = "공선성은 해소됐지만 DW가 아직 낮아 추가 동적 모형 검토가 필요합니다."
    elif abs(delta_aic) < 2 and candidate_max_vif < 4:
        verdict = "단순화 가능"
        reason = "적합 손실이 크지 않으면서 공선성이 해소됐습니다."
    elif delta_aic >= 2 and candidate_max_vif < 4:
        verdict = "제거 비권장"
        reason = "VIF는 정리됐지만 AIC 손실이 커서 설명력 희생이 너무 큽니다."
    elif delta_aic >= 2:
        verdict = "제거 비권장"
        reason = "AIC도 악화되고 공선성 이득도 충분하지 않습니다."
    else:
        verdict = "추가 검토"
        reason = "일부 개선은 있지만 최종 채택 기준을 만족하지 못했습니다."

    return delta_aic, delta_bic, verdict, reason


# S2 이후 추가 실험 셀에서 그대로 재사용할 수 있도록 별칭을 맞춥니다.
run_ols = run_ols_s2


## S1. 전체 구간 베이스라인 OLS

S1은 **모든 시점**을 그대로 넣은 레벨 OLS입니다. 이 단계의 목적은 변수 후보를 빠르게 훑어보고, 어디에서 자기상관과 다중공선성이 심한지 확인하는 것입니다.

여기서 특히 조심해야 할 점은 다음과 같습니다.

- 펌프 정지 구간까지 포함되면 회귀가 물리 메커니즘보다 on/off 상태를 배우기 쉽습니다.
- 시계열 추세가 강하면 `R2`가 높아도 `DW`가 매우 낮을 수 있습니다.
- `max_vif > 10`이면 계수 해석이 불안정하므로, S1 결과는 진단용으로만 보는 편이 안전합니다.


In [ ]:
# S1 실행: 전체 시점을 그대로 사용한 레벨 OLS입니다.
input_path_s1 = resolve_existing_path(INPUT_PATHS)
output_s1 = resolve_output_path(OUTPUT_S1_NAME)
df_base = load_source_frame(input_path_s1)
df_s1 = add_derived_columns(df_base)

s1_rows = []
for target, predictors in TARGET_DICTIONARY.items():
    s1_rows.append(run_ols_s1(df_s1, target, predictors))

s1_results = pd.DataFrame(s1_rows)
s1_results.to_csv(output_s1, index=False)

print(f"불러온 원본 데이터: {input_path_s1.resolve()}")
print(f"S1 저장 경로: {output_s1.resolve()}")
print(f"df_s1 shape: {df_s1.shape}")
display(s1_results[["target", "N", "R2", "AdjR2", "DW", "max_vif", "quality_flags"]])


## S2. 펌프 작동 구간 + HAC + 1차 차분

S2는 S1보다 한 단계 더 보수적인 점검입니다.

이번에는 다음 판단을 반영합니다.

1. 펌프 작동 구간은 `H2_preregistered_spec.md`에 적어둔 H2 분석 설계 기준에 맞춰 **`pump_rpm > 100`** 을 기본 기준으로 사용합니다.
2. 참고로 `flow_rate_l_min > 1.0`과 거의 같지만, 시작 직후 몇 분은 `pump_rpm`은 켜져 있고 유량은 아직 1.0 미만일 수 있습니다. 이 불일치도 같이 기록합니다.
3. `LEVEL_on`은 펌프 작동 구간만 남긴 레벨 회귀이고, `DIFF_on`은 같은 구간을 1차 차분한 회귀입니다.
4. `DIFF_on`에서 `DW`가 2에 가까워지고 `VIF`가 줄어들면, 그쪽이 추세에 덜 끌린 결과라고 볼 수 있습니다.


In [ ]:
# S2 준비: 원본, 펌프 작동 구간, 차분 프레임을 만듭니다.
input_path_s2 = resolve_existing_path(INPUT_PATHS)
df_raw = load_source_frame(input_path_s2)
on_mask, pump_rule_diag, mismatch_preview = diagnose_pump_on_rules(df_raw)
df_on = add_derived_columns(df_raw.loc[on_mask].copy()).set_index("timestamp")
df_diff = df_on.diff().dropna()

print("S2 펌프 작동 구간 진단")
display(pd.DataFrame([pump_rule_diag]))
if not mismatch_preview.empty:
    print("pump_rpm 기준과 flow_rate 기준이 다른 시작 구간 미리보기")
    display(mismatch_preview)

print(f"df_raw  : {df_raw.shape}")   # 기대: (129591, 45)
print(f"df_on   : {df_on.shape}")    # 기대: (65610, 48)
print(f"df_diff : {df_diff.shape}")  # 기대: (65609, 48)


In [ ]:
# S2 실행: 이 셀만 다시 돌리면 됩니다.
output_s2 = resolve_output_path(OUTPUT_S2_NAME)
s2_rows = []
for target, predictors in TARGET_DICTIONARY.items():
    level_row = run_ols_s2(df_on, target, predictors, tag="LEVEL_on")
    diff_row = run_ols_s2(df_diff, target, predictors, tag="DIFF_on")
    if level_row is not None:
        s2_rows.append(level_row)
    if diff_row is not None:
        s2_rows.append(diff_row)

s2_results = pd.DataFrame(s2_rows)
s2_results.to_csv(output_s2, index=False)

print(f"S2 저장 경로: {output_s2.resolve()}  rows={len(s2_results)}")
display(s2_results[["target", "variant", "N", "R2", "AdjR2", "DW", "max_vif", "quality_flags"]])


In [ ]:
# S1과 S2를 나란히 비교해 어떤 결과를 우선 해석할지 추천합니다.
comparison_rows = []
for target in TARGET_DICTIONARY:
    s1_row = s1_results.loc[s1_results["target"] == target].iloc[0].to_dict()
    level_row = s2_results.loc[(s2_results["target"] == target) & (s2_results["variant"] == "LEVEL_on")].iloc[0].to_dict()
    diff_row = s2_results.loc[(s2_results["target"] == target) & (s2_results["variant"] == "DIFF_on")].iloc[0].to_dict()
    recommended_variant, recommendation_reason = recommend_variant(level_row, diff_row)

    comparison_rows.append({
        "target": target,
        "S1_DW": s1_row["DW"],
        "LEVEL_on_DW": level_row["DW"],
        "DIFF_on_DW": diff_row["DW"],
        "S1_max_vif": s1_row["max_vif"],
        "LEVEL_on_max_vif": level_row["max_vif"],
        "DIFF_on_max_vif": diff_row["max_vif"],
        "권장 해석 버전": recommended_variant,
        "권장 이유": recommendation_reason,
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)


## S3. `motor_current_a` / `wire_to_water_efficiency` VIF 정리

S3는 `S2`의 `DIFF_on` 결과를 다시 좁혀 보는 단계입니다.

여기서의 판단은 단순합니다.

1. `LEVEL_on`은 시간 추세 오염이 여전히 커서 더 이상 주력 비교 대상으로 쓰지 않습니다.
2. `DIFF_on`에서도 `motor_current_a`, `wire_to_water_efficiency`는 쌍공선 변수쌍이 남아 있습니다.
3. 그래서 한 번에 한 변수씩 제거한 후보 모델 두 개씩을 만들고, `v2_base`와 `AIC/BIC`, `DW`, `max_vif`를 같이 비교합니다.
   - 이름 읽는 법: `motor_current_a__v3a_drop_discharge` = `motor_current_a` 타깃에서 `v3a` 후보이며 `discharge_pressure_kpa`를 뺀 모델
4. 핵심은 **VIF가 낮아졌다는 이유만으로 채택하지 않는다**는 점입니다. 설명력 손실이 너무 크면 제거는 정당화되지 않습니다.
5. 현재 데이터에서는 두 타깃 모두 축소 모델이 `AIC`를 크게 악화시켜, 최종 판정은 `제거 비권장`입니다.


In [ ]:
# S3 대상: DIFF_on에서도 공선성이 큰 두 타깃만 다시 정리합니다.
# 이름 읽는 법
# - <target>__v2_base            : S2의 DIFF_on 기본 모델
# - <target>__v3a_drop_xxx      : S3의 첫 번째 축소 후보, xxx 변수를 제거한 모델
# - <target>__v3b_drop_xxx      : S3의 두 번째 축소 후보, xxx 변수를 제거한 모델
# 예) motor_current_a__v3b_drop_rpm
#   -> motor_current_a 타깃
#   -> S3의 두 번째 후보(v3b)
#   -> pump_rpm을 제거한 모델
TARGET_DICTIONARY_V3 = {
    # motor_current_a
    # v2 base: motor_temperature_c, pump_rpm, discharge_pressure_kpa
    # DIFF_on VIF: motor_temp≈1.02, pump_rpm≈62.7, discharge_pressure≈62.6
    "motor_current_a__v3a_drop_discharge": [
        "motor_temperature_c",
        "pump_rpm",
    ],
    "motor_current_a__v3b_drop_rpm": [
        "motor_temperature_c",
        "discharge_pressure_kpa",
    ],

    # wire_to_water_efficiency
    # v2 base: pump_rpm, motor_temperature_c, suction_pressure_kpa, bearing_temperature_c
    # DIFF_on VIF: pump_rpm≈28.6, motor_temp≈1.24, suction_pressure≈28.5, bearing_temp≈1.24
    "wire_to_water_efficiency__v3a_drop_suction": [
        "pump_rpm",
        "motor_temperature_c",
        "bearing_temperature_c",
    ],
    "wire_to_water_efficiency__v3b_drop_rpm": [
        "motor_temperature_c",
        "suction_pressure_kpa",
        "bearing_temperature_c",
    ],
}


VARIANT_LABELS_S3 = {
    "motor_current_a__v2_base": "motor_current_a 기본모델(S2 DIFF_on)",
    "motor_current_a__v3a_drop_discharge": "motor_current_a 후보 A: discharge_pressure_kpa 제거",
    "motor_current_a__v3b_drop_rpm": "motor_current_a 후보 B: pump_rpm 제거",
    "wire_to_water_efficiency__v2_base": "wire_to_water_efficiency 기본모델(S2 DIFF_on)",
    "wire_to_water_efficiency__v3a_drop_suction": "wire_to_water_efficiency 후보 A: suction_pressure_kpa 제거",
    "wire_to_water_efficiency__v3b_drop_rpm": "wire_to_water_efficiency 후보 B: pump_rpm 제거",
}


def base_target(key):
    """후보 모델 이름에서 실제 타깃명을 분리합니다."""
    return key.split("__")[0]


def describe_s3_variant(key):
    """variant 문자열을 한국어 설명으로 바꿉니다."""
    return VARIANT_LABELS_S3.get(key, key)


# 필요한 중간 결과가 메모리에 없으면 S2 산출물을 다시 불러옵니다.
if "df_diff" not in globals():
    input_path_s3 = resolve_existing_path(INPUT_PATHS)
    df_raw_s3 = load_source_frame(input_path_s3)
    on_mask_s3, _, _ = diagnose_pump_on_rules(df_raw_s3)
    df_diff = add_derived_columns(df_raw_s3.loc[on_mask_s3].copy()).set_index("timestamp").diff().dropna()

if "s2_results" not in globals():
    s2_results = pd.read_csv(resolve_existing_output_path(OUTPUT_S2_NAME))


# S3는 S2의 DIFF_on 데이터와 HAC OLS 러너를 그대로 재사용합니다.
rows = []
for key, predictors in TARGET_DICTIONARY_V3.items():
    target = base_target(key)
    result = run_ols(df_diff, target, predictors, tag=key)
    if result is not None:
        rows.append(result)

s3_results = pd.DataFrame(rows)
s3_results["variant_설명"] = s3_results["variant"].map(describe_s3_variant)

# 비교 기준은 S2의 DIFF_on 기본 모델입니다.
base_s3 = s2_results[
    (s2_results["variant"] == "DIFF_on")
    & (s2_results["target"].isin(["motor_current_a", "wire_to_water_efficiency"]))
].copy()
base_s3["variant"] = base_s3["target"] + "__v2_base"

comparison_rows_s3 = []
for target in ["motor_current_a", "wire_to_water_efficiency"]:
    base_row = base_s3.loc[base_s3["target"] == target].iloc[0].to_dict()
    comparison_rows_s3.append({
        "variant": base_row["variant"],
        "variant_설명": describe_s3_variant(base_row["variant"]),
        "target": target,
        "N": base_row["N"],
        "R2": base_row["R2"],
        "AdjR2": base_row["AdjR2"],
        "DW": base_row["DW"],
        "AIC": base_row["AIC"],
        "BIC": base_row["BIC"],
        "max_vif": base_row["max_vif"],
        "delta_AIC_vs_base": 0.0,
        "delta_BIC_vs_base": 0.0,
        "판정": "기준 모델",
        "판정 이유": "비교 기준이 되는 S2 DIFF_on 기본 모델입니다.",
    })

    for _, candidate in s3_results.loc[s3_results["target"] == target].iterrows():
        delta_aic, delta_bic, verdict, reason = judge_s3_candidate(base_row, candidate)
        comparison_rows_s3.append({
            "variant": candidate["variant"],
            "variant_설명": describe_s3_variant(candidate["variant"]),
            "target": target,
            "N": candidate["N"],
            "R2": candidate["R2"],
            "AdjR2": candidate["AdjR2"],
            "DW": candidate["DW"],
            "AIC": candidate["AIC"],
            "BIC": candidate["BIC"],
            "max_vif": candidate["max_vif"],
            "delta_AIC_vs_base": delta_aic,
            "delta_BIC_vs_base": delta_bic,
            "판정": verdict,
            "판정 이유": reason,
        })

output_s3 = resolve_output_path(OUTPUT_S3_NAME)
s3_comparison = pd.DataFrame(comparison_rows_s3).sort_values(["target", "AIC"]).reset_index(drop=True)
s3_comparison.to_csv(output_s3, index=False)

print("\n===== S3 비교표 (AIC 오름차순 = 상대적으로 유리) =====")
print(s3_comparison.to_string(index=False))
print(f"\n저장: {output_s3.resolve()}")
display(s3_results[["variant", "variant_설명", "target", "N", "R2", "AdjR2", "DW", "AIC", "BIC", "max_vif", "pvals", "vif"]])


## S5. lag feature 스캔으로 자기상관 다시 점검

S5는 `S2`와 `S3` 이후에도 `DW`가 충분히 올라오지 않은 타깃을 대상으로 진행합니다.

여기서는 `DIFF_on` 데이터만 사용합니다. `LEVEL_on`은 이미 시간 추세와 상태전이 영향이 커서, lag를 붙여도 구조 해석이 더 혼란스러워질 수 있기 때문입니다.

이번 단계에서 가장 중요하게 보는 것은 세 가지입니다.

1. `DW`가 2에 얼마나 가까워졌는가
2. lag 때문에 빠진 행을 **같이 맞춘 기준 모델**과 비교했을 때 `AIC/BIC`가 실제로 좋아졌는가
3. `VIF`가 다시 과도하게 커지지 않았는가

특히 2번이 중요합니다. lag `k`를 넣으면 맨 앞 `k`행이 빠지므로, `S2`의 전체 `DIFF_on` base와 AIC를 바로 비교하면 완전히 공정하지 않습니다. 그래서 S5에서는 **각 후보 모델마다 같은 시점만 남긴 base 모델을 다시 적합**해서 `delta_AIC_vs_matched_base`를 계산합니다.

variant 이름 읽는 법도 여기서 같이 정리합니다.

- `<target>__s5_base` : 현재 시점 설명변수만 쓴 기준 모델
- `<target>__s5_predlag_only_lag1` : 현재 변수는 빼고, 설명변수의 1분 전 값만 사용
- `<target>__s5_current_plus_predlag_lag1` : 현재 설명변수 + 1분 전 설명변수
- `<target>__s5_current_plus_targetlag_lag1` : 현재 설명변수 + 1분 전 타깃
- `<target>__s5_current_plus_predlag_targetlag_lag1` : 현재 설명변수 + 1분 전 설명변수 + 1분 전 타깃

샘플링 간격이 1분이므로 `lag1`, `lag3`, `lag5`, `lag10`은 각각 1분, 3분, 5분, 10분 시차를 뜻합니다.


In [ ]:
import importlib
import sys

# S5 셀만 다시 실행해도 되도록 필요한 입력을 먼저 확인합니다.
if "df_diff" not in globals():
    input_path_s5 = resolve_existing_path(INPUT_PATHS)
    df_raw_s5 = load_source_frame(input_path_s5)
    on_mask_s5, _, _ = diagnose_pump_on_rules(df_raw_s5)
    df_diff = add_derived_columns(df_raw_s5.loc[on_mask_s5].copy()).set_index("timestamp").diff().dropna()

analysis_home = get_analysis_home()
if str(analysis_home) not in sys.path:
    sys.path.insert(0, str(analysis_home))

import s5_lag_scan

# helper 파일을 수정한 뒤 이 셀을 다시 돌려도 최신 코드가 반영되도록 reload합니다.
importlib.reload(s5_lag_scan)
from s5_lag_scan import run_s5_lag_scan

# S5는 zone1_resistance를 제외한 4개 타깃만 다시 봅니다.
S5_TARGETS = [
    "motor_current_a",
    "wire_to_water_efficiency",
    "mix_ph",
    "mix_ec_ds_m",
]

s5_scan, s5_summary = run_s5_lag_scan(
    df_diff=df_diff,
    target_dictionary=TARGET_DICTIONARY,
    target_notes=TARGET_NOTES,
    targets=S5_TARGETS,
    output_scan_path=resolve_output_path(OUTPUT_S5_SCAN_NAME),
    output_summary_path=resolve_output_path(OUTPUT_S5_SUMMARY_NAME),
)

print("===== S5 추천 요약 =====")
print(
    s5_summary[
        [
            "target",
            "variant",
            "variant_설명",
            "N",
            "DW",
            "AIC",
            "max_vif",
            "delta_AIC_vs_matched_base",
            "dw_improvement_vs_matched_base",
            "판정",
        ]
    ].to_string(index=False)
)
print(f"\nS5 전체 스캔 저장: {resolve_output_path(OUTPUT_S5_SCAN_NAME).resolve()}")
print(f"S5 추천 요약 저장: {resolve_output_path(OUTPUT_S5_SUMMARY_NAME).resolve()}")

display(
    s5_scan.sort_values(["target", "판정_우선순위", "delta_AIC_vs_matched_base", "dw_distance_to_2"])[
        [
            "target",
            "variant",
            "variant_설명",
            "N",
            "DW",
            "AIC",
            "max_vif",
            "delta_AIC_vs_matched_base",
            "dw_improvement_vs_matched_base",
            "판정",
            "판정 이유",
            "predictors",
        ]
    ]
)
